## DECK & CARDS CLASSES

In [ ]:
import random
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

class Deck:
    def __init__(self):
        # initialize the deck with 92 cards, 8 cards of each number for each color, 1 card of 0 for each color, 4 cards of Skip for each color
        self.cards = []
        for color in ['Red', 'Green', 'Blue', 'Yellow']:
            for number in range(1,10):
                for _ in range(2):    # 2 copies per color
                    strNum = str(number)
                    self.cards.append(Card(color,strNum))
            self.cards.append(Card(color,'0'))    # 1 0 for each color
            for i in range(4):
                self.cards.append(Card(color, "Skip")) # 4 cards of Skip for each color
        # shuffle the deck
        random.shuffle(self.cards)

    def draw_card(self):
        return self.cards.pop()


## NECESSARY GAME FUNCTIONS

In [ ]:
def valid_move(card, top_card):
    if(card.value == top_card.value):
        return True
    if(card.color == top_card.color):
        return True
    return False

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        x = valid_move(card, top_card)
        if(x == True):
            valid_moves.append(card)
    return valid_moves


def deal_cards(deck, num_players=3, cards_per_player=5):
    hands = []
    for i in range(num_players):
        hand = []  # this for each player, end par we will append to hands list
        for j in range(cards_per_player):
            hand.append(deck.draw_card())
        hands.append(hand)
    return hands

def initial_state():
    deck = Deck()                           # creates and shuffles 92 cards
    hands = deal_cards(deck)               # deal 5 cards to each player
    top_card = deck.cards.pop()            # flip top card to start

    # make sure top card is not a skip
    while(top_card.value == "Skip"):
        deck.cards.append(top_card)        # put skip back
        random.shuffle(deck.cards)
        top_card = deck.cards.pop()

    state = {
        'hands': hands,                    # [[p1 cards], [p2 cards], [p3 cards]]
        'top_card': top_card,
        'deck': deck.cards,                
        'current_player': 0,              # 0=player#1, 1=player#2, 2=player#3
            }
    return state

def apply_move(state, move):
    # Manual deepcopy becuz deepcopy nhi kaam karta
    new_hands = []
    for h in state['hands']:
        new_hands.append(h.copy())

    new_state = {
        'hands': new_hands,
        'top_card': state['top_card'],
        'deck': state['deck'].copy(),
        'current_player': state['current_player']
    }
    current = new_state['current_player']
    hand = new_state['hands'][current]  # get current player's hand
    top_card = new_state['top_card']
    deck = new_state['deck']
    valid_moves = get_valid_moves(hand,top_card)
    if(len(valid_moves) == 0): 
        #deck empty then pass turn
        if len(deck) > 0:
            hand.append(deck.pop())
        new_player = (current + 1) % 3
    else:
        if move is None:
            raise ValueError("move cannot be None when valid moves exist")
        hand.remove(move)
        new_state['top_card'] = move
        if(move.value == "Skip"):
            new_player = (current + 2) % 3
        else:
            new_player = (current + 1) % 3
    new_state['current_player'] = new_player
    return new_state


def evaluation(state, player_index, strategy, num_players=3):
    player_hand = state['hands'][player_index]
    COPP = 0
    CAI = len(player_hand)
    S = 0
    for opponent in range(num_players):
        if(opponent == player_index):   # skip player index
            continue 
        opponent_hand = state['hands'][opponent]
        COPP += len(opponent_hand)
    for card in player_hand:
        if(card.value == "Skip"):
            S += 1

    COPP = COPP / (num_players - 1)   # since COPP is average num cards in oppoenets so num_players - 1
    if (strategy == "defensive"):
        return 50 - 7*CAI + 2*COPP + 4*S
    elif(strategy == "offensive"):                                  
        return 50 - 5*CAI + 3*COPP + 2*S


## MINIMAX & EXPECTIMAX FUNCTIONS

In [ ]:
def is_terminal(state):
    for hand in state['hands']:
        if len(hand) == 0:      # someone won
            return True
    if len(state['deck']) == 0: # deck empty
        return True
    return False


def minimax(state, depth, is_maximizing, player_index):
    # base case
    if depth == 0 or is_terminal(state):
        return evaluation(state, player_index, "defensive")
    
    current = state['current_player']
    hand = state['hands'][current]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)
    
    if len(valid_moves) == 0:
        moves = [None]          # only move is to draw
    else:
        moves = valid_moves

    if is_maximizing:           # P1's turn -> maximize score
        best_score = float('-inf')
        for move in moves:
            new_state = apply_move(state, move)
            score = minimax(new_state, depth-1, False, player_index)
            best_score = max(best_score, score)
        return best_score

    else:                       # opponent's turn -> minimize score
        best_score = float('inf')
        for move in moves:
            new_state = apply_move(state, move)
            score = minimax(new_state, depth-1, True, player_index)
            best_score = min(best_score, score)
        return best_score


def expectimax(state, depth, player_index):
    # base case
    if depth == 0 or is_terminal(state):
        return evaluation(state, player_index, "offensive")
    
    current = state['current_player']
    hand = state['hands'][current]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if current == player_index:             # MAX node -> P2's turn
        if len(valid_moves) == 0:
            new_state = apply_move(state, None)
            return expectimax(new_state, depth-1, player_index)
        
        best_score = float('-inf')
        for move in valid_moves:
            new_state = apply_move(state, move)
            score = expectimax(new_state, depth-1, player_index)
            best_score = max(best_score, score)
        return best_score

    elif len(valid_moves) == 0:             # CHANCE node -> opponent must draw
        deck = state['deck']
        if len(deck) == 0:
            return evaluation(state, player_index, "offensive")
        
        total_score = 0
        for draw_card in deck:                          # consider each possible draw
            prob = 1 / len(deck)                        # equal probability for each card
            #manual copy becuz deepcopy nhi kaam karta
            new_hands = []
            for h in state['hands']:
                new_hands.append(h.copy())
            new_state = {
                'hands': new_hands,
                'top_card': state['top_card'],
                'deck': state['deck'].copy(),
                'current_player': state['current_player']
            }
            new_state['hands'][current].append(draw_card)
            new_state['deck'].remove(draw_card)
            new_state['current_player'] = (current + 1) % 3
            score = expectimax(new_state, depth-1, player_index)
            total_score += prob * score
        return total_score

    else:                                   # opponent turn -> random legal move
        scores = []
        for move in valid_moves:
            new_state = apply_move(state, move)
            score = expectimax(new_state, depth-1, player_index)
            scores.append(score)
        return sum(scores) / len(scores)    # average over all opponent moves


def get_best_move_minimax(state, player_index):
    hand = state['hands'][player_index]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if len(valid_moves) == 0:
        return None             # must draw

    best_score = float('-inf')
    best_move = None

    for move in valid_moves:
        new_state = apply_move(state, move)
        score = minimax(new_state, depth=3, is_maximizing=False, player_index=player_index)
        print(f"  Play: {move.color} {move.value} -> Score: {score}")
        if score > best_score:
            best_score = score
            best_move = move

    # Print the decision tree for the chosen move (why it was selected)
    if best_move is not None:
        print("\nDecision tree for chosen move (Minimax):")
        print(f"Chosen: Play {best_move.color} {best_move.value} (root score={best_score})")
        child = apply_move(state, best_move)
        # After P1 plays, it's opponent's turn => MIN node
        minimax_trace(child, depth=2, is_maximizing=False, player_index=player_index, indent="  ", max_branch=6)

    return best_move


def get_best_move_expectimax(state, player_index):
    hand = state['hands'][player_index]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if len(valid_moves) == 0:
        return None                         # must draw

    best_score = float('-inf')
    best_move = None

    for move in valid_moves:
        new_state = apply_move(state, move)
        score = expectimax(new_state, depth=3, player_index=player_index)
        print(f"  Play: {move.color} {move.value} -> Expected Score: {score:.2f}")
        if score > best_score:
            best_score = score
            best_move = move

    if best_move is not None:
        print("\nDecision tree for chosen move (Expectimax):")
        print(f"Chosen: Play {best_move.color} {best_move.value} (root expected={best_score:.2f})")
        child = apply_move(state, best_move)
        expectimax_trace(child, depth=2, player_index=player_index, indent="  ", max_branch=6, max_chance=6)

    return best_move


## GAME TREE FUNCTIONS

In [ ]:
def print_turn_decision_tree(player_name, scored_actions, chosen_index, depth2_lines=None):
    print("\nDecision Tree (this turn):")
    print(player_name)

    if len(scored_actions) == 0:
        print("  (no actions)")
        return

    for i, (action_text, value) in enumerate(scored_actions):
        mark = "*" if i == chosen_index else " "
        print(f"  {mark} {action_text} -> {value:.2f}")

        if depth2_lines is not None and action_text in depth2_lines:
            for line in depth2_lines[action_text]:
                print(f"      {line}")


def explain_minimax(state_after_root_move, focus_player, max_opp=4, max_next=4):

    lines = []

    opp = state_after_root_move['current_player']
    opp_hand = state_after_root_move['hands'][opp]
    opp_top = state_after_root_move['top_card']
    opp_moves = get_valid_moves(opp_hand, opp_top)

    # If opponent cannot play, only option is Draw
    if len(opp_moves) == 0:
        opp_actions = [None]
    else:
        opp_actions = opp_moves

    shown_opp = opp_actions[:max_opp]
    option_values = []

    lines.append(f"Opponent P{opp+1} options (MIN):")

    for om in shown_opp:
        action_text = "Draw" if om is None else f"Play {om.color} {om.value}"
        reply_state = apply_move(state_after_root_move, om)

        nxt = reply_state['current_player']
        nxt_hand = reply_state['hands'][nxt]
        nxt_top = reply_state['top_card']
        nxt_moves = get_valid_moves(nxt_hand, nxt_top)

        # Next layer is MAX
        if len(nxt_moves) == 0:
            nxt_actions = [None]
        else:
            nxt_actions = nxt_moves

        shown_nxt = nxt_actions[:max_next]
        leaf_scores = []

        lines.append(f"  - {action_text} -> Next P{nxt+1} options (MAX):")
        for nm in shown_nxt:
            nm_text = "Draw" if nm is None else f"Play {nm.color} {nm.value}"
            leaf_state = apply_move(reply_state, nm)
            leaf = evaluation(leaf_state, focus_player, "defensive")
            leaf_scores.append(leaf)
            lines.append(f"      {nm_text} -> leaf {leaf:.2f}")

        if len(nxt_actions) > max_next:
            lines.append(f"      ... {len(nxt_actions) - max_next} more moves not shown")

        max_pick = max(leaf_scores) if len(leaf_scores) > 0 else evaluation(reply_state, focus_player, "defensive")
        option_values.append(max_pick)
        lines.append(f"    MAX picks {max_pick:.2f}")

    if len(opp_actions) > max_opp:
        lines.append(f"  ... {len(opp_actions) - max_opp} more opponent moves not shown")

    min_pick = min(option_values) if len(option_values) > 0 else evaluation(state_after_root_move, focus_player, "defensive")
    lines.append(f"MIN picks {min_pick:.2f}")
    return lines, min_pick


def explain_expectimax(state_after_root_move, focus_player, max_opp=4, max_next=4, max_draw=4):

    lines = []

    opp = state_after_root_move['current_player']
    opp_hand = state_after_root_move['hands'][opp]
    opp_top = state_after_root_move['top_card']
    opp_moves = get_valid_moves(opp_hand, opp_top)

    # If opponent has no moves: CHANCE draw node
    if len(opp_moves) == 0:
        deck = state_after_root_move['deck']
        lines.append(f"Chance node: P{opp+1} must draw (deck size {len(deck)})")
        if len(deck) == 0:
            leaf = evaluation(state_after_root_move, focus_player, "offensive")
            lines.append(f"deck empty -> leaf {leaf:.2f}")
            return lines, leaf

        shown = deck[:max_draw]
        vals = []
        for dc in shown:
            # manual copy since copy.deepcopy() doesn't work for some reason with dictionary idk why
            new_hands = []
            for h in state_after_root_move['hands']:
                new_hands.append(h.copy())
            new_state = {
                'hands': new_hands,
                'top_card': state_after_root_move['top_card'],
                'deck': state_after_root_move['deck'].copy(),
                'current_player': state_after_root_move['current_player']
            }
            new_state['hands'][opp].append(dc)
            new_state['deck'].remove(dc)
            new_state['current_player'] = (opp + 1) % 3

            leaf = evaluation(new_state, focus_player, "offensive")
            vals.append(leaf)
            lines.append(f"  - draw {dc.color} {dc.value} -> leaf {leaf:.2f}")

        if len(deck) > max_draw:
            lines.append(f"  ... {len(deck) - max_draw} more draws not shown")

        approx_e = sum(vals) / len(vals) if len(vals) > 0 else 0.0
        lines.append(f"Approx E = {approx_e:.2f}")
        return lines, approx_e

    lines.append(f"Opponent P{opp+1} options (RANDOM, averaged):")
    shown_opp = opp_moves[:max_opp]
    opp_vals = []

    for om in shown_opp:
        reply_state = apply_move(state_after_root_move, om)
        nxt = reply_state['current_player']
        nxt_hand = reply_state['hands'][nxt]
        nxt_top = reply_state['top_card']
        nxt_moves = get_valid_moves(nxt_hand, nxt_top)

        if len(nxt_moves) == 0:
            nxt_actions = [None]
        else:
            nxt_actions = nxt_moves

        shown_nxt = nxt_actions[:max_next]
        leaf_scores = []

        lines.append(f"  - Play {om.color} {om.value} -> Next P{nxt+1} options:")
        for nm in shown_nxt:
            nm_text = "Draw" if nm is None else f"Play {nm.color} {nm.value}"
            leaf_state = apply_move(reply_state, nm)
            leaf = evaluation(leaf_state, focus_player, "offensive")
            leaf_scores.append(leaf)
            lines.append(f"      {nm_text} -> leaf {leaf:.2f}")

        if len(nxt_actions) > max_next:
            lines.append(f"      ... {len(nxt_actions) - max_next} more moves not shown")

        # summarize by average of shown leaves.
        avg_leaf = sum(leaf_scores) / len(leaf_scores) if len(leaf_scores) > 0 else evaluation(reply_state, focus_player, "offensive")
        opp_vals.append(avg_leaf)
        lines.append(f"    AVG (shown) = {avg_leaf:.2f}")

    if len(opp_moves) > max_opp:
        lines.append(f"  ... {len(opp_moves) - max_opp} more opponent moves not shown")

    approx_avg = sum(opp_vals) / len(opp_vals) if len(opp_vals) > 0 else 0.0
    lines.append(f"Approx OPP average = {approx_avg:.2f}")
    return lines, approx_avg


def step_simulation(state, verbose=False, p3_mode="simulation", p3_manual_move=None):
    current = state['current_player']

    # Winner check
    for i, h in enumerate(state['hands']):
        if len(h) == 0:
            return state, f"P{i+1} already won.", True, i

    if current == 0:
        move = get_best_move_minimax(state, 0, verbose=verbose)
        actor = "P1"
    elif current == 1:
        move = get_best_move_expectimax(state, 1, verbose=verbose)
        actor = "P2"
    else:
        actor = "P3"
        if p3_mode == "manual":
            hand = state['hands'][2]
            top_card = state['top_card']
            valid = get_valid_moves(hand, top_card)
            if len(valid) == 0:
                move = None
            else:
                move = p3_manual_move
                if move is None:
                    raise ValueError("P3 manual mode needs p3_manual_move when a valid move exists")
        else:
            # Simulation mode: P3 uses minimax logic
            move = get_best_move_minimax(state, 2, verbose=verbose)

    if move is None:
        msg = f"{actor} draws"
    else:
        msg = f"{actor} plays {move.color} {move.value}"

    new_state = apply_move(state, move)

    # Winner check (after move)
    for i, h in enumerate(new_state['hands']):
        if len(h) == 0:
            return new_state, msg, True, i

    # Deck empty check
    if len(new_state['deck']) == 0:
        return new_state, "Deck empty. Game over.", True, None

    return new_state, msg, False, None


## INTERFACE TO RUN THE CODE

In [ ]:
def pretty_card(card):
    return f"{card.color} {card.value}"

def print_state_simple(state, turn_no):
    print("\n" + "="*70)
    print(f"Turn {turn_no} | Current Player: P{state['current_player']+1}")
    print(f"Top Card: {pretty_card(state['top_card'])}")
    print(f"P1 hand ({len(state['hands'][0])}): " + ", ".join(pretty_card(c) for c in state['hands'][0]))
    print(f"P2 hand ({len(state['hands'][1])}): " + ", ".join(pretty_card(c) for c in state['hands'][1]))
    print(f"P3 hand ({len(state['hands'][2])}): " + ", ".join(pretty_card(c) for c in state['hands'][2]))
    print(f"Deck remaining: {len(state['deck'])}")

def choose_manual_p3_move(state):
    hand = state['hands'][2]
    top_card = state['top_card']
    valid = get_valid_moves(hand, top_card)

    if len(valid) == 0:
        print("P3 manual: no valid move, will draw.")
        return None

    print("\nP3 manual turn. Choose one card index to play:")
    for i, c in enumerate(valid):
        print(f"  [{i}] {c.color} {c.value}")

    while True:
        raw = input("Enter index: ").strip()
        if raw.isdigit():
            idx = int(raw)
            if 0 <= idx < len(valid):
                return valid[idx]
        print("Invalid index. Try again.")

mode = input("Choose mode for P3 ('manual' or 'simulation'): ").strip().lower()
if mode not in ("manual", "simulation"):
    print("Invalid choice. Defaulting to simulation.")
    mode = "simulation"

state = initial_state()
turn = 1
max_turns = 200

print_state_simple(state, turn_no=0)

while turn <= max_turns:
    if mode == "manual" and state['current_player'] == 2:
        manual_move = choose_manual_p3_move(state)
    else:
        manual_move = None

    new_state, msg, game_over, winner = step_simulation(
        state,
        verbose=True,
        p3_mode=mode,
        p3_manual_move=manual_move,
    )
    print(f"\nAction: {msg}")
    state = new_state
    print_state_simple(state, turn_no=turn)

    if game_over:
        if winner is None:
            print("\nGame over: deck empty.")
        else:
            print(f"\nWinner: P{winner+1}")
        break

    turn += 1

if turn > max_turns:
    print("\nStopped at max_turns.")
